# 02 - Limpeza e Tratamento

Objetivo deste notebook:
- limpar e padronizar cada dataset bruto por fonte
- manter nomes de campos em pt-br
- preparar cada base para integracao futura por `codigo_municipio + ano`

Padrao de organizacao:
- funcoes auxiliares no inicio
- uma secao por fonte de dados
- cada fonte deve gerar um dataframe tratado proprio


## 1. Imports, caminhos e configuracao

In [ ]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from IPython.display import display

PASTA_DATASETS = Path('../datasets')
PASTA_IDH = PASTA_DATASETS / 'idh'
PASTA_CRIMES = PASTA_DATASETS / 'crimes'
PASTA_POPULACAO = PASTA_DATASETS / 'populacao'
PASTA_EDUCACAO = PASTA_DATASETS / 'educacao'
PASTA_AUXILIARES = PASTA_DATASETS / 'auxiliares'

ANO_REFERENCIA_IDH = 2010
CAMINHO_TABELA_MUNICIPIOS = PASTA_AUXILIARES / 'codigos_municipios_ibge.csv'

CONFIG_CSV_PADRAO = {
    'sep': ';',
    'encoding': 'utf-8-sig',
    'decimal': ',',
}


## 2. Funcoes auxiliares de limpeza

In [ ]:
def listar_csvs(pasta: Path) -> list[Path]:
    if not pasta.exists():
        return []
    return sorted(pasta.glob('*.csv'))


def ler_csv_padrao(caminho: Path) -> pd.DataFrame:
    df = pd.read_csv(caminho, **CONFIG_CSV_PADRAO)
    return df.dropna(axis=1, how='all')


def normalizar_texto(texto: str) -> str:
    if pd.isna(texto):
        return None
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(caractere for caractere in texto if not unicodedata.combining(caractere))
    texto = re.sub(r'\s+', ' ', texto)
    return texto


def converter_colunas_numericas(df: pd.DataFrame, colunas: list[str]) -> pd.DataFrame:
    resultado = df.copy()
    for coluna in colunas:
        resultado[coluna] = (
            resultado[coluna]
            .astype(str)
            .str.replace(',', '.', regex=False)
            .pipe(pd.to_numeric, errors='coerce')
        )
    return resultado


def validar_base(df: pd.DataFrame, chave: list[str], nome_dataset: str) -> None:
    print(f'Dataset tratado: {nome_dataset}')
    print(f'Linhas: {df.shape[0]}')
    print(f'Colunas: {df.shape[1]}')
    print(f'Duplicatas exatas: {df.duplicated().sum()}')
    print(f'Duplicatas na chave {chave}: {df.duplicated(subset=chave).sum()}')
    display(df.isna().sum().to_frame('qtd_nulos'))


# Fonte 1 - IDH Municipal 2010

## 3.1 Leitura do IDH bruto

In [ ]:
arquivos_idh = listar_csvs(PASTA_IDH)
ARQUIVO_IDH = arquivos_idh[0]
ARQUIVO_IDH


In [ ]:
df_idh_raw = ler_csv_padrao(ARQUIVO_IDH)
df_idh_raw.head()


## 3.2 Padronizacao de colunas do IDH

In [ ]:
mapa_colunas_idh = {
    'Territorialidades': 'territorialidade',
    'Territorialidade': 'territorialidade',
    'IDHM 2010': 'idhm',
    'IDHM': 'idhm',
    'IDHM Renda 2010': 'idhm_renda',
    'IDHM Renda': 'idhm_renda',
    'IDHM Educação 2010': 'idhm_educacao',
    'IDHM Educação': 'idhm_educacao',
    'IDHM Longevidade 2010': 'idhm_longevidade',
    'IDHM Longevidade': 'idhm_longevidade',
}

df_idh = df_idh_raw.rename(columns=mapa_colunas_idh).copy()
df_idh.columns.tolist()


## 3.3 Criar campos municipais do IDH

In [ ]:
df_idh['ano'] = ANO_REFERENCIA_IDH
        
df_idh['sigla_uf'] = df_idh['territorialidade'].astype(str).str.extract(r'\(([A-Z]{2})\)$')[0]
df_idh['nome_municipio'] = df_idh['territorialidade'].astype(str).str.replace(r'\s*\([A-Z]{2}\)$', '', regex=True)
df_idh = df_idh[df_idh['sigla_uf'].notna()].copy()

df_idh['nome_municipio_padronizado'] = df_idh['nome_municipio'].apply(normalizar_texto)
df_idh['sigla_uf'] = df_idh['sigla_uf'].astype(str).str.strip().str.upper()

display(df_idh[['territorialidade', 'nome_municipio', 'nome_municipio_padronizado', 'sigla_uf']].head(10))


## 3.4 Criar codigo da UF no padrao IBGE

In [ ]:
mapa_codigo_uf_ibge = {
    'RO': 11, 'AC': 12, 'AM': 13, 'RR': 14, 'PA': 15, 'AP': 16, 'TO': 17,
    'MA': 21, 'PI': 22, 'CE': 23, 'RN': 24, 'PB': 25, 'PE': 26, 'AL': 27,
    'SE': 28, 'BA': 29, 'MG': 31, 'ES': 32, 'RJ': 33, 'SP': 35, 'PR': 41,
    'SC': 42, 'RS': 43, 'MS': 50, 'MT': 51, 'GO': 52, 'DF': 53,
}

df_idh['codigo_uf_ibge'] = df_idh['sigla_uf'].map(mapa_codigo_uf_ibge)
df_idh['codigo_municipio'] = pd.NA

display(df_idh[['nome_municipio', 'sigla_uf', 'codigo_uf_ibge', 'codigo_municipio']].head(10))


## 3.5 Ajustar tipos e colunas finais do IDH

In [ ]:
colunas_numericas_idh = [
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
    'ano',
    'codigo_uf_ibge',
]

df_idh = converter_colunas_numericas(df_idh, colunas_numericas_idh)

colunas_idh_ordenadas = [
    'codigo_municipio',
    'codigo_uf_ibge',
    'sigla_uf',
    'nome_municipio',
    'nome_municipio_padronizado',
    'ano',
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
    'territorialidade',
]

df_idh = df_idh[colunas_idh_ordenadas].copy()
display(df_idh.head())


## 3.6 Cruzamento opcional para adicionar codigo_municipio

In [ ]:
if CAMINHO_TABELA_MUNICIPIOS.exists():
    df_codigos = pd.read_csv(CAMINHO_TABELA_MUNICIPIOS)
    df_codigos['nome_municipio_padronizado'] = df_codigos['nome_municipio_padronizado'].apply(normalizar_texto)
    df_codigos['sigla_uf'] = df_codigos['sigla_uf'].astype(str).str.upper().str.strip()

    df_idh = df_idh.drop(columns=['codigo_municipio']).merge(
        df_codigos[['codigo_municipio', 'sigla_uf', 'nome_municipio_padronizado']],
        how='left',
        on=['sigla_uf', 'nome_municipio_padronizado'],
        validate='one_to_one'
    )

    print('Municipios sem codigo_municipio:', df_idh['codigo_municipio'].isna().sum())
else:
    print(f'Tabela auxiliar nao encontrada em: {CAMINHO_TABELA_MUNICIPIOS}')
    print('A base de IDH foi preparada para receber o codigo_municipio em uma proxima etapa.')


## 3.7 Validacao final do IDH

In [ ]:
validar_base(df_idh, ['nome_municipio', 'sigla_uf', 'ano'], 'IDH municipal 2010')
display(df_idh.head())


# Fonte 2 - Crimes

## 4.1 Leitura dos crimes brutos

In [ ]:
arquivos_crimes = listar_csvs(PASTA_CRIMES)
if arquivos_crimes:
    ARQUIVO_CRIMES = arquivos_crimes[0]
    df_crimes_raw = ler_csv_padrao(ARQUIVO_CRIMES)
    display(df_crimes_raw.head())
else:
    print('Nenhum CSV de crimes encontrado em datasets/crimes.')


## 4.2 Tratamento dos crimes

Este bloco sera preenchido depois da exploracao completa do dataset de crimes.
Resultado esperado desta fonte:
- `codigo_municipio`
- `ano`
- indicadores de criminalidade agregados em `municipio + ano`


In [ ]:
# df_crimes_tratado = ...


# Fonte 3 - Populacao

In [ ]:
arquivos_populacao = listar_csvs(PASTA_POPULACAO)
if arquivos_populacao:
    ARQUIVO_POPULACAO = arquivos_populacao[0]
    df_populacao_raw = ler_csv_padrao(ARQUIVO_POPULACAO)
    display(df_populacao_raw.head())
else:
    print('Nenhum CSV de populacao encontrado em datasets/populacao.')


Tratamento futuro esperado para populacao:
- `codigo_municipio`
- `ano`
- `populacao`
- possivel `crescimento_populacional_pct`


# Fonte 4 - Educacao

In [ ]:
arquivos_educacao = listar_csvs(PASTA_EDUCACAO)
if arquivos_educacao:
    ARQUIVO_EDUCACAO = arquivos_educacao[0]
    df_educacao_raw = ler_csv_padrao(ARQUIVO_EDUCACAO)
    display(df_educacao_raw.head())
else:
    print('Nenhum CSV de educacao encontrado em datasets/educacao.')


Tratamento futuro esperado para educacao:
- `codigo_municipio`
- `ano`
- indicadores educacionais selecionados


# Saidas esperadas deste notebook

Ao final desta etapa, o projeto deve ter dataframes tratados por fonte:
- `df_idh`
- `df_crimes_tratado`
- `df_populacao_tratado`
- `df_educacao_tratado`

Esses dataframes entram no notebook 03 para integracao e feature engineering.
